# Test FBG Sensitivity Analysis Script
Este notebook prueba el script `fbg_press_sensitivity_analysis.py` como módulo de Python.

In [ ]:
# Import the analysis module
import sys
sys.path.append('/eos/user/v/vgarciap/SWAN_projects/Sensitivity (Press.)')

from fbg_press_sensitivity_analysis import run_analysis

print("✓ Module imported successfully!")

## Test 1: Auto-detect plateaus
Detecta automáticamente los plateaus desde los datos de presión.

In [ ]:
# Run analysis with auto-plateau detection
filepath = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20250310.root"

results = run_analysis(
    filepath=filepath,
    auto_plateaus=True,
    tolerance=0.01,
    min_plateau_length=500,
    time_shift_hours=1.0
)

## Test 2: Manual plateaus
Define los plateaus manualmente.

In [ ]:
# Run analysis with manual plateau times
plateau_times = [
    ("Plateau 1", "14:50", "15:13"),
    ("Plateau 2", "15:30", "15:52"),
    ("Plateau 3", "16:05", "16:30"),
    ("Plateau 4", "16:40", "16:57"),
    ("Plateau 5", "17:06", "17:12"),
]

results = run_analysis(
    filepath=filepath,
    plateau_times=plateau_times,
    time_shift_hours=1.0,
    temp_sensors=["RTD-7", "RTD-8"]
)

## Test 3: Only RTD-7 reference
Usa solo RTD-7 como referencia de temperatura.

In [ ]:
# Run analysis with only RTD-7
results_rtd7 = run_analysis(
    filepath=filepath,
    auto_plateaus=True,
    tolerance=0.01,
    min_plateau_length=500,
    time_shift_hours=1.0,
    temp_sensors=["RTD-7"]
)

## Access individual results
Puedes acceder a los resultados individuales para análisis adicional.

In [ ]:
# Example: Get sensitivity for FBG-1-P vs RTD-7
fbg_sensor = "FBG-1-P"
temp_sensor = "RTD-7"

if fbg_sensor in results and temp_sensor in results[fbg_sensor]:
    sens_data = results[fbg_sensor][temp_sensor]
    
    print(f"\nDetailed results for {fbg_sensor} vs {temp_sensor}:")
    print(f"  Sensitivity: {sens_data['slope_pm_K']:.3f} pm/K")
    print(f"  R²: {sens_data['r_squared']:.6f}")
    print(f"  P-value: {sens_data['p_value']:.2e}")
    print(f"  Std Error: {sens_data['std_err']:.3f}")
    print(f"  Intercept: {sens_data['intercept_pm']:.2f} pm")

## Test 4: Comparar diferentes parámetros de auto-detección

Prueba cómo diferentes valores de tolerancia afectan la detección de plateaus.

In [ ]:
# Probar diferentes tolerancias
print("="*80)
print("COMPARACIÓN DE PARÁMETROS DE AUTO-DETECCIÓN")
print("="*80)

# Importar find_plateaus para inspección manual
from fbg_press_sensitivity_analysis import load_root_data, convert_timestamps, find_plateaus

# Cargar datos
data = load_root_data(filepath)
timestamps = convert_timestamps(data["times"])

# Probar diferentes tolerancias
tolerances = [0.005, 0.01, 0.02]

for tol in tolerances:
    plateaus = find_plateaus(data["press"], timestamps, tolerance=tol, min_plateau_length=500)
    print(f"\nTolerancia = {tol*100}%:")
    print(f"  Plateaus detectados: {len(plateaus)}")
    if plateaus:
        for idx, p in enumerate(plateaus):
            print(f"    P{idx+1}: {p['t0'].strftime('%H:%M')} - {p['tfin'].strftime('%H:%M')} "
                  f"(P={p['mean']:.2f}±{p['std']:.3f} bar)")

## Test 5: Análisis rápido - Solo tabla

Esta es la forma más rápida de obtener solo la tabla de resultados sin más output.

In [ ]:
# Análisis directo con los plateaus del experimento del 10-03-2025
plateau_times_experiment = [
    ("Plateau 1", "14:50", "15:13"),
    ("Plateau 2", "15:30", "15:52"),
    ("Plateau 3", "16:05", "16:30"),
    ("Plateau 4", "16:40", "16:57"),
    ("Plateau 5", "17:06", "17:12"),
]

results_final = run_analysis(
    filepath="/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20250310.root",
    plateau_times=plateau_times_experiment,
    time_shift_hours=1.0,
    temp_sensors=["RTD-7", "RTD-8"]
)

print("\n🎉 ¡Análisis completo! La tabla de sensibilidades se muestra arriba.")